In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/reemalharthii/train-videos/train_videos.csv
/kaggle/input/datasets/reemalharthii/test-videos/test_videos.csv
/kaggle/input/datasets/reemalharthii/sample-submission/sample_submission.csv


**1: Imports and Setup**

In [22]:
import os
import warnings
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')

Libraries loaded successfully!


**2: Load Datasets**

In [23]:
# Locate and Load Datasets Automatically
input_dir = '/kaggle/input'
train_path, test_path = None, None

for root, dirs, files in os.walk(input_dir):
  if 'train_videos.csv' in files:
    train_path = os.path.join(root, 'train_videos.csv')
  if 'test_videos.csv' in files:
    test_path = os.path.join(root, 'test_videos.csv')

if not train_path or not test_path:
  raise FileNotFoundError(
      'Could not find train_videos.csv or test_videos.csv in /kaggle/input'
  )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f'Train loaded shape: {train_df.shape}')
print(f'Test loaded shape: {test_df.shape}')

Train loaded shape: (12000, 27)
Test loaded shape: (3001, 26)


**3: Feature Engineering Function**

In [24]:
def create_robust_features(df):
  df = df.copy()

  view_cols = [
      c
      for c in df.columns
      if 'view' in c.lower() and 'target' not in c.lower()
  ]
  like_cols = [c for c in df.columns if 'like' in c.lower()]
  share_cols = [c for c in df.columns if 'share' in c.lower()]
  comment_cols = [c for c in df.columns if 'comment' in c.lower()]

  # Totals & Basic Stats
  if view_cols:
    df['total_views_0_5'] = df[view_cols].sum(axis=1)
    df['last_day_views'] = df[view_cols[-1]]
    df['first_day_views'] = df[view_cols[0]]
    df['mean_views'] = df[view_cols].mean(axis=1)
    df['median_views'] = df[view_cols].median(axis=1)
    df['std_views'] = df[view_cols].std(axis=1).fillna(0)

  if like_cols and view_cols:
    df['total_likes'] = df[like_cols].sum(axis=1)
    df['like_ratio'] = df['total_likes'] / (df['total_views_0_5'] + 1)

  if share_cols and view_cols:
    df['total_shares'] = df[share_cols].sum(axis=1)
    df['share_ratio'] = df['total_shares'] / (df['total_views_0_5'] + 1)

  if comment_cols and view_cols:
    df['total_comments'] = df[comment_cols].sum(axis=1)
    df['comment_ratio'] = df['total_comments'] / (df['total_views_0_5'] + 1)

  # Growth Indicators
  if len(view_cols) >= 2:
    df['growth_last_vs_first'] = (df[view_cols[-1]] - df[view_cols[0]]) / (
        df[view_cols[0]] + 1
    )
    df['growth_last_vs_prev'] = (df[view_cols[-1]] - df[view_cols[-2]]) / (
        df[view_cols[-2]] + 1
    )

  # Convert Object Categoricals to Categorical Type for LGBM
  cat_cols = df.select_dtypes(include=['object']).columns
  for c in cat_cols:
    df[c] = df[c].astype('category')

  return df


train_feats = create_robust_features(train_df)
test_feats = create_robust_features(test_df)
print('Features created successfully!')

Features created successfully!


**4: Prepare Data & Target Transformation**

In [25]:
# Identify target and ID columns
target_col = [
    c for c in train_df.columns if 'target' in c.lower() or '30' in c
][0]
id_col = 'id' if 'id' in train_feats.columns else train_feats.columns[0]

features = [c for c in train_feats.columns if c not in [id_col, target_col]]

X = train_feats[features]
y = train_feats[target_col]
X_test = test_feats[features]

# Fill NAs for numeric features for Ridge
X_num = X.select_dtypes(include=[np.number]).fillna(0)
X_test_num = X_test.select_dtypes(include=[np.number]).fillna(0)

# Target log transformation
y_log = np.log1p(np.maximum(0, y))

print(f'Training with {len(features)} features.')

Training with 25 features.


**5: Cross-Validation & Model Training (LightGBM)**

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_preds = np.zeros(len(test_feats))
ridge_preds = np.zeros(len(test_feats))

print('Starting 5-Fold Training...')

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
  X_tr, y_tr_log = X.iloc[train_idx], y_log.iloc[train_idx]
  X_va, y_va_log = X.iloc[val_idx], y_log.iloc[val_idx]

  # Model 1: LightGBM Regressor
  lgb = LGBMRegressor(
      n_estimators=1000,
      learning_rate=0.015,
      num_leaves=15,
      max_depth=4,
      min_child_samples=50,
      subsample=0.7,
      colsample_bytree=0.7,
      reg_alpha=1.0,
      reg_lambda=5.0,
      random_state=42 + fold,
      n_jobs=-1,
  )
  lgb.fit(X_tr, y_tr_log)
  lgb_preds += np.expm1(np.maximum(0, lgb.predict(X_test))) / kf.n_splits

  # Model 2: Ridge Linear Regressor
  X_tr_num = X_num.iloc[train_idx]
  ridge = Ridge(alpha=100.0)
  ridge.fit(X_tr_num, y_tr_log)
  ridge_preds += (
      np.expm1(np.maximum(0, ridge.predict(X_test_num))) / kf.n_splits
  )

  print(f'Fold {fold + 1} completed.')

print('All folds finished training!')

Starting 5-Fold Training...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001488 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4052
[LightGBM] [Info] Number of data points in the train set: 9600, number of used features: 23
[LightGBM] [Info] Start training from score 7.129470
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

**6: Generate Submission CSV File**

In [29]:
# 1. Blend 70% LightGBM + 30% Ridge
blended_preds = 0.7 * lgb_preds + 0.3 * ridge_preds

# 2. Get last day view column dynamically from test_df
view_cols = [
    c
    for c in test_df.columns
    if 'view' in c.lower() and 'target' not in c.lower()
]

if 'last_day_views' in test_feats.columns:
  day5_views = test_feats['last_day_views'].values
elif view_cols:
  day5_views = test_df[view_cols[-1]].values
else:
  day5_views = blended_preds

# 3. Aggressive Outlier Clipping (Strict Capping)
# Limit predictions based on percentile AND absolute threshold to eliminate RMSE spikes
percentile_cap = np.percentile(blended_preds, 95.0)
ratio_cap = day5_views * 15.0  # Max 15x growth from day 5

# Apply absolute max cap at 35,000 views
final_predictions = np.minimum(blended_preds, ratio_cap)
final_predictions = np.minimum(final_predictions, percentile_cap)
final_predictions = np.minimum(
    final_predictions, 35000.0
)  # Hard limit for outliers
final_predictions = np.maximum(0, final_predictions)

# 4. Save Final Submission File
submission = pd.DataFrame(
    {id_col: test_feats[id_col], target_col: final_predictions}
)

submission.to_csv('submission.csv', index=False)

print('=' * 50)
print('SUCCESS! submission.csv created with optimized predictions.')
print(
    f'Prediction Stats -> Min: {final_predictions.min():.2f}, Mean:'
    f' {final_predictions.mean():.2f}, Max: {final_predictions.max():.2f}'
)
print('=' * 50)

SUCCESS! submission.csv created with optimized predictions.
Prediction Stats -> Min: 215.77, Mean: 1712.75, Max: 6821.50
